# Apply pipelines
We combine *preprocessing* and *classification* into a pipeline, then perform grid search to find the best classifier in the *classification* stage of the pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import mglearn

## 1. Revisiting adult dataset from week 5 - Encoding example

In [ ]:
import os
# The file has no headers naming the columns, so we pass header=None
# and provide the column names explicitly in "names"
adult_path = os.path.join(mglearn.datasets.DATA_PATH, "adult.data")
data = pd.read_csv(
    adult_path, header=None, index_col=False,
    names=['age', 'workclass', 'fnlwgt', 'education',  'education-num',
           'marital-status', 'occupation', 'relationship', 'race', 'gender',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
           'income'])
# For illustration purposes, we only select some of the columns
data = data[['age', 'workclass', 'education', 'gender', 'hours-per-week',
             'occupation', 'income']]
# IPython.display allows nice output formatting within the Jupyter notebook
display(data.head())

In [ ]:
data.info()

### 1.1 Build the ColumnTransformer
**Transform by column name**

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ct = ColumnTransformer(
    [("scaling", StandardScaler(), ['age', 'hours-per-week']),
     ("onehot", OneHotEncoder(sparse_output=False), ['workclass', 'education', 'gender', 'occupation'])])

**We can select columns by data type too.**

In [ ]:
ct = ColumnTransformer(
    [("scaling", StandardScaler(), make_column_selector(dtype_exclude=object)),
     ("onehot", OneHotEncoder(sparse_output=False), make_column_selector(dtype_include=object))])

### 1.2 Build the Pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[('preprocessor', ct),
                      ('classifier', LogisticRegression(max_iter=1000))])

In [ ]:
from sklearn import set_config
set_config(display='diagram')
pipe

### 1.3 Split the data

In [ ]:
from sklearn.model_selection import train_test_split
# get all columns apart from income for the features
data_features = data.drop("income", axis=1)
# split dataframe and income
X_train, X_test, y_train, y_test = train_test_split(
    data_features, data.income, random_state=0)



### 1.4 Train the pipeline and assess on training set


In [ ]:
pipe.fit(X_train, y_train)

print("Training score: {:.2f}".format(pipe.score(X_train, y_train)))

### 1.5 Access pipeline steps

In [ ]:
set_config(display='diagram')
pipe.named_steps.preprocessor

In [ ]:
set_config(display='diagram')
pipe['preprocessor']

In [ ]:
set_config(display='text')
pipe['preprocessor'].transformers_[1][1].get_feature_names_out(['workclass', 'education', 'gender', 'occupation'])

In [ ]:
pipe.named_steps.classifier.get_params()

### 1.6 Grid search
We use the named step in the pipeline, followed by two underscores, then the hyper-parameter name to define the grid parameters:

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'classifier__C': [0.01, 0.1, 1.0, 10.0],
             'classifier__fit_intercept': [True, False]}
grid = GridSearchCV(pipe, param_grid, cv=5)


In [ ]:
grid.fit(X_train, y_train)

print("Best params:\n{}\n".format(grid.best_params_))
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))